# MouseFull matched ablation study

Joint **D1 + D3** ablations with eleven separate anatomical classes, the existing SHAM/TBI classifier and the corrected partial-label ROI loss. The reference configuration matches notebook 05's MouseFull task and original-width 3DR-Net settings. Use the **rnet** kernel and inspect annotation coverage before enabling execution.

The default variants independently remove residual shortcuts, attention, multiscale inputs, decoder-head fusion, or the classifier, and compare strided downsampling. Each changes one setting; the reference is unchanged. All variants use the same registered animals, split, preprocessing, loss, maximum epoch budget and checkpoint-selection rule. Execution delegates to the shared ablation runner.
`K_FOLD_COMPATIBLE=False` keeps fixed-split training. Set it to `True` to run every outer fold with independent holdouts and save one checkpoint per fold. No additional permanent test split is created. CV uses LR reduction without early stopping by default; viewers follow the selected evaluation role.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Complete MouseFull reference configuration

The historical eleven-class channel order is preserved. Contra/ipsilateral corpus callosum, ventricles, hippocampus and cortex remain separate; extra anatomy outside the selected mapping is projected to task background. Availability follows normalized metadata: missing extended anatomy becomes **-1** targets. Verified lesion-free D3 SHAM scans always use available empty lesion targets, as in the original pipeline. `dice_ce_gradient_fix` consumes logits during fitting; exports expose full softmax probabilities. Its documented positive-available-target policy does not establish complete background annotation.

The reference includes the SHAM/TBI classifier. Architecture and dropout defaults are unchanged. Every variant uses TBI backbone initialization with a fresh optimizer by default. Provide a matching TBI source for each changed backbone; the no-classifier, no-fusion and loss variants can reuse the reference backbone. D2 and external D4 are excluded. Use `animal_ids` for confirmed cross-cohort biological duplicates; identities are not guessed from similar original IDs.

`MODE="fixed_split"` uses common train/validation/test animals. `MODE="cross_validation"` uses five grouped outer folds with separate checkpoint-selection validation. Default execution fits seven models for one fixed split/seed, or thirty-five for five-fold CV. Actual epochs can differ with early stopping. Seeds are matched without promising bitwise augmentation replay. This notebook performs no interpolation sensitivity analysis.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


In [ ]:
from typing import Literal

from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig
from src.experiments.initialization import cv_initializations_from_run

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
K_FOLD_COMPATIBLE: bool = False  # True runs every fold for each study member.
MODE: Literal["fixed_split", "cross_validation"] = "fixed_split"
if K_FOLD_COMPATIBLE:
    MODE = "cross_validation"
TRAINING_SEEDS: tuple[int, ...] = (42,)
RUN_STUDY: bool = False
USE_TBI_PRETRAINING: bool = True
TBI_RUN: Path | None = None  # Reference TBI source; must not fit target validation or use target test animals.
TBI_FOLD: int = 0
FREEZE_BACKBONE: bool = False
TBI_FOLD_SOURCES: dict[int, tuple[Path, int]] = {}  # Reference target fold -> source run/fold.
TBI_VARIANT_RUNS: dict[str, tuple[Path, int]] = {}  # Variant name -> matching source run/fold.
TBI_VARIANT_FOLD_SOURCES: dict[str, dict[int, tuple[Path, int]]] = {}  # Variant -> target fold -> source run/fold.
RESUME_STUDY: bool = False
STUDY_DIR: Path = REPO_ROOT / "results" / "ablations" / "mouse_full_3drnet_components"
SPLIT_FILE: Path | None = None
FOLD_FILE: Path | None = None

training = TrainingConfig(
    model_name="mouse_full_3drnet_ablation", k=5, epochs=300, batch_size=8,
    seed=TRAINING_SEED, data_multiplier=3, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
datasets = (
    DatasetSpec(name="D1", annotation_scope="full_volume"),
    # All D3 SHAM scans are verified lesion-free: supervise an empty lesion mask.
    DatasetSpec(name="D3", annotation_scope="full_volume"),
)
animal_ids: dict[str, str] = {}
# Fractions of all animals; saved split files keep their exact memberships.
split_settings = FixedSplitConfig(
    validation_fraction=0.2, test_fraction=0.1,
)

data_settings = DataConfig(root=DATA_ROOT, datasets=datasets, animal_ids=animal_ids)
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.0,
)

# Preserve the historical MouseFull channel order; keep each side separate.
label_mapping: dict[int, int] = {
    0: 0, 1: 1, 2: 2, 3: 3, 5: 4, 6: 5,
    12: 6, 13: 7, 15: 8, 16: 9, 21: 10,
}
class_names: tuple[str, ...] = (
    "Background", "Lesion", "Contra corpus callosum", "Contra ventricle",
    "Contra hippocampus", "Contra cortex", "Ipsi corpus callosum",
    "Ipsi ventricle", "Ipsi hippocampus", "Ipsi cortex", "Third ventricle",
)
labels = LabelConfig(mapping=label_mapping, names=class_names, unknown="background")
# Sample once; validation/test reuse evaluation_patches.json across epochs and runs.
evaluation_patch_settings = EvaluationPatchConfig(
    patches_per_scan=3, sampling="center_biased", seed=42,
    left_right_pairs=False,  # D1/rats: optional paired LR views; MouseFull: keep False.
)

model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    multiscale=True, deep_supervision=True,
    residual=True, attention=True, max_pooling=True, classifier=True,
)
loss_settings = LossConfig(
    regions="dice_ce_gradient_fix", gradient_fix_version=1, gamma=0.8, brain_alpha=0.6, batch_wise=True,
    # Total loss = weighted sum of the active output-head losses.
    regions_weight=1.0, brain_weight=1.0, classifier_weight=1.0,
)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)
initialization = InitializationConfig()
if USE_TBI_PRETRAINING and TBI_RUN is not None:
    initialization = InitializationConfig.from_run(
        run_dir=TBI_RUN, fold=TBI_FOLD, freeze_backbone=FREEZE_BACKBONE,
    )

initialization_by_fold: dict[int, InitializationConfig] = {}
if USE_TBI_PRETRAINING and K_FOLD_COMPATIBLE and not TBI_FOLD_SOURCES and TBI_RUN is not None:
    initialization_by_fold = cv_initializations_from_run(
        run_dir=TBI_RUN, k=training.k, freeze_backbone=FREEZE_BACKBONE,
    )
elif USE_TBI_PRETRAINING:
    for target_fold, (source_run, source_fold) in TBI_FOLD_SOURCES.items():
        initialization_by_fold[target_fold] = InitializationConfig.from_run(
            run_dir=source_run, fold=source_fold, freeze_backbone=FREEZE_BACKBONE,
        )

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    evaluation_patches=evaluation_patch_settings,
    fixed_split=split_settings,
    model=model_settings, labels=labels, loss=loss_settings,
    inference=inference_settings, initialization=initialization,
    initialization_by_fold=initialization_by_fold,
    learning_rate=0.001, split_seed=42,
    cv_validation_fraction=0.2,
)
config.validate()

## Configure callbacks directly

`MODE="cross_validation"` uses only LR reduction. Fixed-split studies add TensorBoard, early stopping and CSV logging. All variants/seeds receive the same list. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", mode="min", factor=0.5, patience=10,
    min_lr=1e-7, verbose=1,
)
callbacks: list[tf.keras.callbacks.Callback] = [reduce_lr]

# K-Fold uses only ReduceLROnPlateau; fixed-split studies add logging and early stopping.
if MODE == "fixed_split":
    log_dir: str = "tensorboard"
    csv_filename: str = "training.csv"
    csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)
    tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=20, verbose=1,
        restore_best_weights=False,  # Best saved weights are restored by the runner.
    )
    callbacks.extend([tensorboard, early_stopping, csv_logger])


## Single-setting experiment matrix

Each variant changes one supported setting. `residual=False` removes shortcuts throughout the network, including any multiscale branches; encoder-decoder skip concatenations remain. `attention=False` removes decoder attention. `max_pooling=False` replaces pooling with strided convolution.

`multiscale=False` removes the three pooled-input feature branches and their encoder concatenations. `deep_supervision=False` removes the coarse projections and their sum from both segmentation heads, retaining each original finest-level projection and final output layer. This ablates fused decoder logits; the original model has no separate auxiliary losses. Both switches default to `True`.

`no_classifier` is enabled by default and removes only the auxiliary condition head. Its lesion false-positive report remains available even though classifier metrics are absent. All other component rows retain the v1 corrected loss and the classifier.

Optional `loss_variants` compare v1 with v3; v2 remains an optional diagnostic after its observed convergence problem. These rows keep the full reference architecture, task weights and verified empty SHAM lesion supervision. Legacy `dice_ce` is not supported for this partial-label data contract. Compare segmentation metrics across objectives; loss values have different meanings.

With TBI pretraining, residual, attention, downsampling and multiscale changes require matching source backbones. No-fusion, no-classifier and loss rows can reuse the reference TBI features; segmentation/classifier heads are freshly initialized in every variant. Each source is checked against target fitting/selection roles; source validation may be shared with target selection-validation, while final holdouts remain independent.

Optional augmentation, cropping and repetition rows change training strategy; repetition changes optimizer updates per epoch. All rows reuse the same saved animals, evaluation patches and seeds. A changed matrix requires a new study folder; resume validates the original matrix. The U-Net comparison belongs in notebook 08.

Fresh fixed studies preserve animal roles from all selected pretraining sources, then fill overall fractions with new animals. Fresh CV studies require aligned per-fold source checkpoints; their held-out source animals must cover the matching cohort exactly once. Source selection-validation is reused only as target selection-validation.


In [ ]:
from IPython.display import display
from src.experiments.ablation import AblationVariant, ablation_plan

variants: list[AblationVariant] = [
    AblationVariant("reference"),
    AblationVariant("no_residual", "residual", False),
    AblationVariant("no_attention", "attention", False),
    AblationVariant("no_multiscale", "multiscale", False),
    AblationVariant("no_deep_supervision", "deep_supervision", False),
    AblationVariant("strided_downsampling", "max_pooling", False),
    AblationVariant("no_classifier", "classifier", False),
    # AblationVariant("no_augmentation", "augment", False),
    # AblationVariant("fixed_center_crop", "random_crop", False),
    # AblationVariant("no_repetition", "data_multiplier", 1),
]
# Loss rows keep every architecture/training setting at the reference value.
loss_variants: list[AblationVariant] = [
    # AblationVariant("gradient_fix_v3", "gradient_fix_version", 3),
    # AblationVariant("gradient_fix_v2", "gradient_fix_version", 2),  # Optional diagnostic; observed poor convergence.
]
variants.extend(loss_variants)

from dataclasses import replace

if USE_TBI_PRETRAINING:
    configured_variants: list[AblationVariant] = []
    for variant in variants:
        variant_initialization = None
        variant_fold_initializations = None
        if variant.name in TBI_VARIANT_RUNS:
            source_run, source_fold = TBI_VARIANT_RUNS[variant.name]
            variant_initialization = InitializationConfig.from_run(
                run_dir=source_run, fold=source_fold, freeze_backbone=FREEZE_BACKBONE,
            )
        if variant.name in TBI_VARIANT_FOLD_SOURCES:
            variant_fold_initializations = {}
            for target_fold, (source_run, source_fold) in TBI_VARIANT_FOLD_SOURCES[variant.name].items():
                variant_fold_initializations[target_fold] = InitializationConfig.from_run(
                    run_dir=source_run, fold=source_fold, freeze_backbone=FREEZE_BACKBONE,
                )
        elif K_FOLD_COMPATIBLE and variant_initialization is not None:
            variant_source_run = variant_initialization.source_experiment.parent
            variant_fold_initializations = cv_initializations_from_run(
                run_dir=variant_source_run, k=training.k, freeze_backbone=FREEZE_BACKBONE,
            )
        configured_variant = replace(
            variant, initialization=variant_initialization,
            initialization_by_fold=variant_fold_initializations,
        )
        configured_variants.append(configured_variant)
    variants = configured_variants

plan = ablation_plan(config, variants)
display(plan)

models_per_variant: int = 1
if MODE == "cross_validation":
    models_per_variant = config.training.k
total_models: int = len(variants) * len(TRAINING_SEEDS) * models_per_variant
print("Maximum fitted models:", total_models)

## One shared MouseFull animal selection

Load notebook 05's saved `splits.json` using `SPLIT_FILE` for a matched fixed-split study. For CV, load five outer folds using `FOLD_FILE`, covering exactly the registered D1+D3 animals. Otherwise generate one unstratified selection. All scans of an animal remain together. Every variant/seed reuses that selection; validation/test scans are neither multiplied nor dropped.

Review dataset, condition, modality and annotation balance before manuscript training. This study's CV pool covers all registered animals; notebook 06 fold files generated with a separately reserved test cohort do not have the same pool.

Set `RESUME_STUDY=True` to recover this notebook's dedicated `STUDY_DIR`. The shared selection helper loads its saved selection, and the runner validates the complete study contract. Keep settings, matrix, seed order and inputs unchanged. Completed runs are reused; fresh studies reject existing folders.

In [ ]:
from src.experiments.data import prepare_data
from src.experiments.ablation import study_selection

prepared = prepare_data(config)
selection_initializations = [variant.configuration(config).initialization for variant in variants]
selection_initializations_by_fold: dict[int, list[InitializationConfig]] = {}
if MODE == "cross_validation":
    for index in range(config.training.k):
        source_settings = []
        for variant in variants:
            variant_config = variant.configuration(config)
            settings = variant_config.initialization_by_fold.get(index, variant_config.initialization)
            source_settings.append(settings)
        selection_initializations_by_fold[index] = source_settings
selection = study_selection(
    data=prepared, mode=MODE, study_dir=STUDY_DIR,
    split_file=SPLIT_FILE, fold_file=FOLD_FILE, resume=RESUME_STUDY,
    initializations=selection_initializations,
    initializations_by_fold=selection_initializations_by_fold,
)
split = selection.split
folds = selection.folds
summary = selection.summary
coverage = prepared.annotation_summary()
manifest = prepared.subjects.manifest(prepared.subjects.subject_ids)
manifest_columns = ["Subject", "Dataset", "OriginalID", "Modality", "Scan"]

display(summary)
display(coverage)
display(manifest[manifest_columns].head())

## Check a common validation batch

Validation is deterministic, keeps its last incomplete batch and is not multiplied. This preview uses the reference model's three target contracts without constructing a model: ROI labels, brain masks and SHAM/TBI condition. Unavailable ROI channels remain -1; classifier and brain supervision remain independent of ROI annotation coverage. The no-classifier ablation omits only the condition target in its own runner configuration.

In [ ]:
import numpy as np

validation_ids: list[str] = []
if split is not None:
    validation_ids = list(split.validation)
else:
    validation_ids = folds[0].val_ids

validation_generator = prepared.make_generator(validation_ids, training=False, batch_size=1)
images, targets = validation_generator[0]
print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Condition targets (SHAM=0, TBI=1):", targets[2].ravel())
print("Validation batches:", len(validation_generator))

unavailable = np.all(targets[0][0] == -1, axis=(0, 1, 2))
missing_names = np.asarray(config.labels.names)[unavailable]
print("Unannotated channels in this scan:", missing_names.tolist())

del images, targets, validation_generator

## Run or resume the matched study

The shared runner compiles each configuration, manages checkpoints and restores validation-selected best weights before full-volume test/outer-holdout evaluation. The study writes its matrix, source/configuration contract, shared split and run index. Each variant/seed retains its own complete experiment artifacts and recovery state. No expensive training starts while `RUN_STUDY=False`.

Predictions include all eleven ROI classes together, independent of annotation availability. Missing reference classes are excluded from measurements, and unavailable/empty-mask statuses remain explicit. Saved ROI and brain masks restore the original scan grid.

In [ ]:
from src.experiments.ablation import run_ablation

study = None
if RUN_STUDY:
    if USE_TBI_PRETRAINING:
        for variant in variants:
            variant_config = variant.configuration(config)
            selected_initializations = [variant_config.initialization]
            if MODE == "cross_validation":
                selected_initializations = []
                for fold in folds:
                    fold_initialization = variant_config.initialization_by_fold.get(fold.index, variant_config.initialization)
                    selected_initializations.append(fold_initialization)
            if any(settings.mode == "scratch" for settings in selected_initializations):
                raise ValueError("Provide TBI sources for each variant/fold, or disable USE_TBI_PRETRAINING")

    study = run_ablation(
        prepared, variants, study_dir=STUDY_DIR, mode=MODE,
        split=split, folds=folds, seeds=TRAINING_SEEDS, resume=RESUME_STUDY, callbacks=callbacks,
    )
    print("Study saved to:", study.study_dir)
else:
    print("Matrix and data checked. Set RUN_STUDY = True to run or resume.")

## Inspect descriptive results

Compare full-volume measurements, statuses, actual training/inference time and complexity before interpreting changes. The plot below uses one scan-average per animal, variant, seed and fold; seed/fold identities remain visible. Repeated models are not extra independent animals. This notebook performs no significance tests or automatic hyperparameter selection; animal-aware paired analyses and uncertainty belong in the revision-results notebook.

`full_volume_metrics.csv` retains raw scan rows and unavailable values. `runs.json` links each variant/seed to its saved configuration. `timings.csv` retains per-fold costs. Predictions and detailed histories remain inside each run folder.

In [ ]:
import plotly.express as px
import plotly.io as pio
from src.experiments.ablation import animal_metric_values

pio.renderers.default = "vscode"
evaluation_role: str = "test"
if MODE == "cross_validation":
    evaluation_role = "holdout"

if study is not None:
    display(study.timings)
    dice = animal_metric_values(study, metric="Dice", role=evaluation_role)
    display(dice)
    if not dice.empty:
        figure = px.box(
            dice, x="Variant", y="Value", color="Variant", facet_col="Region",
            points="all", hover_data=["Subject", "Seed", "Fold", "Scans"],
            title="MouseFull full-volume Dice by animal and model",
        )
        figure.show()

## Best-checkpoint patch and full-volume measurements

Native patch losses and metrics use every saved deterministic validation/test patch. These are cohort aggregates, not independent animal scores. Full-volume brain and region scores use postprocessed predictions, original annotation IDs and valid references. The workbook contains separate sheets for the two evaluation stages.


In [ ]:
import pandas as pd

if study is not None:
    display(study.patch_metrics)
    full_volume_metrics = study.metrics
    evaluation_folder = study.study_dir
    selected_columns = ["DatasetRole", "Fold", "Head", "Region", "Metric", "Value", "Status"]
    if not full_volume_metrics.empty:
        display(full_volume_metrics[selected_columns].head(30))
    from src.evaluation.classification import classification_report, healthy_lesion_report
    classifier_report_result = classification_report(study.classifier_predictions)
    if not classifier_report_result.summary.empty:
        display(classifier_report_result.summary)
        display(classifier_report_result.confusion)
    sham_animals, sham_summary = healthy_lesion_report(full_volume_metrics)
    if not sham_summary.empty:
        display(sham_summary)
    print("Complete native patch and postprocessed volume tables:", evaluation_folder)
    print("Workbook:", evaluation_folder / "evaluation_metrics.xlsx")
else:
    print("Use the final best-checkpoint cell to evaluate a stopped or saved run.")


## Explore test predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.

`VIEW_MEMBER` selects a variant/seed from the printed study run keys. The first member is the default. Fixed-split studies display test scans; CV studies display outer holdouts.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_MEMBER: str | None = None  # For example, a key printed below.

viewer_role = "test"
if MODE == "cross_validation":
    viewer_role = "holdout"

viewer_run = VIEW_RUN
if viewer_run is None and study is not None:
    print("Available study runs:", list(study.runs))
    viewer_member = VIEW_MEMBER
    if viewer_member is None:
        viewer_member = next(iter(study.runs))
    viewer_run = study.runs[viewer_member].run_dir

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, study mode and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This measures native patches and performs full-volume inference, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = "test"  # Use holdout for a CV run after a fresh kernel start.
CHECKPOINT_MEMBER: str | None = None  # Available variant/seed key, if study exists above.
if globals().get("MODE") == "cross_validation":
    CHECKPOINT_ROLE = "holdout"

checkpoint_run = CHECKPOINT_RUN
current_study = globals().get("study")
if checkpoint_run is None and current_study is not None:
    print("Available study runs:", list(current_study.runs))
    checkpoint_member = CHECKPOINT_MEMBER
    if checkpoint_member is None:
        checkpoint_member = next(iter(current_study.runs))
    checkpoint_run = current_study.runs[checkpoint_member].run_dir

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
        evaluate_patches=True,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    display(checkpoint_evaluation.patch_metrics)
    print("Evaluation workbook:", checkpoint_evaluation.output_dir / "evaluation_metrics.xlsx")
    from src.evaluation.classification import classification_report, healthy_lesion_report
    checkpoint_classifier = classification_report(checkpoint_evaluation.classifier_predictions)
    if not checkpoint_classifier.summary.empty:
        display(checkpoint_classifier.summary)
        display(checkpoint_classifier.confusion)
    checkpoint_sham_animals, checkpoint_sham_summary = healthy_lesion_report(checkpoint_evaluation.metrics)
    if not checkpoint_sham_summary.empty:
        display(checkpoint_sham_summary)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")
